## 8.0 Introduction

In Lecture 7 we built up matrix algebra from scratch — how matrices multiply, what they do geometrically, and how to compute determinants and eigenvalues. This lecture puts those tools to work on a concrete problem: **linear dynamical systems**, where a matrix describes how a system evolves step by step through time.

We will model the spread of an epidemic through a population using four compartments. Starting from a fully susceptible population, we will simulate the system forward, visualize its evolution, and then use the eigenvalues of the transition matrix to explain *why* the system behaves the way it does — and how fast it converges to its long-run steady state.

In the final section, we modify the model in two ways and observe how the structure of the transition matrix determines the qualitative behavior of the epidemic.

**By the end of this lecture you will be able to:**

- Write and simulate the update equation $\mathbf{x}_{t+1} = A\mathbf{x}_t$
- Build a transition matrix from a word description of system dynamics
- Identify absorbing states and explain their role in the long-run behavior
- Connect the eigenvalues of $A$ to the convergence rate of the system
- Modify a transition matrix and predict how the changes will affect the simulation

```python
import numpy as np
import matplotlib.pyplot as plt
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 8.1 A Linear Dynamical System: The SIRD Epidemic Model

### The Update Equation

Many real-world systems evolve through time according to a simple rule: the state at the next time step is a linear function of the current state. If we encode the system's condition as a vector $\mathbf{x}_t \in \mathbb{R}^n$, this rule takes the form:

$$\mathbf{x}_{t+1} = A\mathbf{x}_t$$

where $A$ is an $n \times n$ **transition matrix** encoding how each component of the state influences the next step. Starting from an initial state $\mathbf{x}_0$, any future state is just repeated matrix multiplication:

$$\mathbf{x}_1 = A\mathbf{x}_0, \quad \mathbf{x}_2 = A^2\mathbf{x}_0, \quad \ldots \quad \mathbf{x}_t = A^t\mathbf{x}_0$$

When the columns of $A$ each sum to 1, the entries of $A$ are probabilities and the system is called a **Markov process**.

### The Four Compartments

We model the spread of a disease through a population using four compartments. At each time step $t$, the state vector is:

$$\mathbf{x}_t = \begin{bmatrix} S_t \\ I_t \\ R_t \\ D_t \end{bmatrix}$$

where each entry is the **fraction of the total population** in that compartment:

- $S_t$: **Susceptible** — healthy, not immune, could get infected
- $I_t$: **Infected** — currently sick and contagious
- $R_t$: **Recovered** — no longer infectious, cannot be reinfected
- $D_t$: **Deceased** — permanently removed from the living population

Because these are fractions of the whole, $S_t + I_t + R_t + D_t = 1$ for all $t$.

The dynamics from step $t$ to $t+1$:

- **Susceptible:** 95% remain susceptible; 5% become infected
- **Infected:** 85% remain infected; 10% recover; 4% return to susceptible (partial immunity loss); 1% die
- **Recovered:** 100% remain recovered — an **absorbing state**
- **Deceased:** 100% remain deceased — an **absorbing state**

**Question.** Why must the columns of $A$ each sum to 1? What would it mean if a column summed to more than 1?

**Question.** $R$ and $D$ are called absorbing states — once you enter, you never leave. What does this tell you about the long-run values of $S$ and $I$?

In [ ]:
# Run this cell without modification — transition diagram
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')

nodes = {
    'S': (2, 4, 'steelblue',  'Susceptible\n$S_t$'),
    'I': (5, 4, 'tomato',     'Infected\n$I_t$'),
    'R': (8, 4, 'seagreen',   'Recovered\n$R_t$'),
    'D': (5, 1, '#888888',    'Deceased\n$D_t$'),
}

for key, (x, y, color, label) in nodes.items():
    circle = plt.Circle((x, y), 0.7, color=color, zorder=3, alpha=0.85)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=8.5,
            fontweight='bold', color='white', zorder=4)

def arrow(ax, x1, y1, x2, y2, label, color='#333333', offset=(0, 0.25)):
    dx, dy = x2-x1, y2-y1
    length = np.sqrt(dx**2+dy**2)
    ux, uy = dx/length, dy/length
    sx, sy = x1+0.72*ux, y1+0.72*uy
    ex, ey = x2-0.72*ux, y2-0.72*uy
    ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.6))
    mx, my = (sx+ex)/2+offset[0], (sy+ey)/2+offset[1]
    ax.text(mx, my, label, ha='center', va='center', fontsize=8,
            color=color, bbox=dict(fc='white', ec='none', pad=1))

def self_loop(ax, x, y, label, color):
    theta = np.linspace(0.2, 2*np.pi-0.2, 100)
    r = 0.55
    lx = x + r*np.cos(theta) + 0.1
    ly = y + r*np.sin(theta) + 0.9
    ax.plot(lx, ly, color=color, lw=1.4, alpha=0.7)
    ax.annotate('', xy=(lx[-1], ly[-1]), xytext=(lx[-2], ly[-2]),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.4))
    ax.text(x+0.7, y+1.55, label, ha='center', fontsize=8,
            color=color, bbox=dict(fc='white', ec='none', pad=1))

arrow(ax, 2, 4, 5, 4, '5% S→I',  'tomato',    offset=(0,  0.3))
arrow(ax, 5, 4, 2, 4, '4% I→S',  'steelblue', offset=(0, -0.3))
arrow(ax, 5, 4, 8, 4, '10% I→R', 'seagreen',  offset=(0,  0.3))
arrow(ax, 5, 4, 5, 1, '1% I→D',  '#888888',   offset=(0.4, 0))
self_loop(ax, 2, 4, '95%',  'steelblue')
self_loop(ax, 5, 4, '85%',  'tomato')
self_loop(ax, 8, 4, '100%', 'seagreen')
self_loop(ax, 5, 1, '100%', '#888888')

ax.set_title('SIRD Transition Diagram', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

### Building the Transition Matrix

Each column $j$ of $A$ describes where compartment $j$ goes at the next step. Entry $a_{ij}$ is the fraction of compartment $j$ that flows into compartment $i$.

$$A = \begin{bmatrix} 0.95 & 0.04 & 0 & 0 \\ 0.05 & 0.85 & 0 & 0 \\ 0 & 0.10 & 1 & 0 \\ 0 & 0.01 & 0 & 1 \end{bmatrix} \begin{matrix} \leftarrow S \\ \leftarrow I \\ \leftarrow R \\ \leftarrow D \end{matrix}$$

Columns 3 and 4 (Recovered, Deceased) have a single 1 on the diagonal — these are the absorbing states.

In [ ]:
A = np.array([[0.95, 0.04, 0,    0],
              [0.05, 0.85, 0,    0],
              [0,    0.10, 1,    0],
              [0,    0.01, 0,    1]])

x0 = np.array([1.0, 0.0, 0.0, 0.0])   # entire population starts susceptible

print('Transition matrix A:')
print(np.round(A, 4))
print()
print('Column sums (should all be 1.0):', np.round(A.sum(axis=0), 6))

### Simulating the Evolution

In [ ]:
T = 200
labels_sird = ['Susceptible', 'Infected', 'Recovered', 'Deceased']
colors_sird  = ['steelblue', 'tomato', 'seagreen', '#888888']

history = np.zeros((4, T+1))
history[:, 0] = x0
x = x0.copy()
for t in range(T):
    x = A @ x
    history[:, t+1] = x

print('State at t = 20:')
for i, name in enumerate(labels_sird):
    print(f'  {name:<15}: {history[i, 20]:.4f}')
print(f'  Sum: {history[:, 20].sum():.6f}')
print()
print('State at t = 100:')
for i, name in enumerate(labels_sird):
    print(f'  {name:<15}: {history[i, 100]:.4f}')

In [ ]:
# Run this cell without modification
t_vals = np.arange(T+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.stackplot(t_vals, history[0], history[1], history[2], history[3],
             labels=labels_sird, colors=colors_sird, alpha=0.85)
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Fraction of population')
ax.set_title('SIRD Model: Population Fractions (stacked)')
ax.legend(loc='center right', fontsize=9)
ax.set_xlim(0, T); ax.set_ylim(0, 1)
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
for i in range(4):
    ax.plot(t_vals, history[i], color=colors_sird[i], lw=2, label=labels_sird[i])
ax.set_xlabel('Time step $t$')
ax.set_ylabel('Fraction of population')
ax.set_title('SIRD Model: Each Compartment')
ax.legend(fontsize=9)
ax.set_xlim(0, T); ax.set_ylim(0, 1)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Epidemic Evolution: SIRD Model', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** The infected curve rises, peaks, then falls. At roughly what time step does it peak? What is happening in the susceptible population at that moment?

**Question.** The stacked chart always has total height 1. Connect this back to the column-sum property of $A$.

### Long-Run Steady State

As $t \to \infty$, the system converges to a steady state $\mathbf{x}_\infty$ that no longer changes: $A\mathbf{x}_\infty = \mathbf{x}_\infty$.

In [ ]:
x_inf = np.linalg.matrix_power(A, 10000) @ x0

print('Long-run steady state:')
for i, name in enumerate(labels_sird):
    print(f'  {name:<15}: {x_inf[i]:.4f}')
print(f'  Sum: {x_inf.sum():.6f}')
print()
print('Verify A @ x_inf = x_inf:', np.allclose(A @ x_inf, x_inf, atol=1e-4))

**Question.** Does the long-run result make sense? Given that $R$ and $D$ are absorbing states, what must the long-run values of $S$ and $I$ approach?

**Question.** Not everyone ends up deceased — roughly 90.9% recover. What determines the split between $R_\infty$ and $D_\infty$?

## 8.2 Eigenvalues, Eigenvectors, and the Steady State

In Lecture 7 you found that eigenvalues are the solutions to $\det(A - \lambda I) = 0$, and that the corresponding eigenvectors satisfy $A\mathbf{v} = \lambda\mathbf{v}$ — multiplying by $A$ only scales them.

The steady state $\mathbf{x}_\infty$ is the most natural example: it satisfies $A\mathbf{x}_\infty = \mathbf{x}_\infty = 1 \cdot \mathbf{x}_\infty$, so it is an eigenvector of $A$ with eigenvalue $\lambda = 1$.

### The Eigenvalues of $A$

In [ ]:
vals, vecs = np.linalg.eig(A)

print('Eigenvalues and their eigenvectors:')
print()
for i in range(4):
    print(f'lambda = {np.real(vals[i]):.6f}')
    print(f'  eigenvector =')
    print(np.round(np.real(vecs[:, i]).reshape(-1, 1), 6))
    print()

Notice that `np.linalg.eig` returns eigenvalues and eigenvectors in matched order: `vals[i]` is the eigenvalue for column `vecs[:, i]`.

$A$ has four eigenvalues: $1,\ 1,\ 0.9671,\ 0.8329$.

The two eigenvectors with $\lambda = 1$ are $[0,0,1,0]$ and $[0,0,0,1]$ — pure Recovered and pure Deceased. These are the **absorbing states**: once the system reaches either, it stays there forever.

The eigenvectors for the non-unit eigenvalues have **negative components**, which means they cannot be interpreted as physical population states. They are mathematical objects that describe how the transient part of the dynamics decays — but we won't try to assign them to specific compartments.

### The $\lambda = 1$ Eigenspace

Because $A$ has *two* eigenvalues equal to 1, the set of all $\lambda=1$ eigenvectors forms a **two-dimensional eigenspace** — any linear combination of $[0,0,1,0]$ and $[0,0,0,1]$ is also a valid $\lambda=1$ eigenvector:

$$\alpha\begin{bmatrix}0\\0\\1\\0\end{bmatrix} + \beta\begin{bmatrix}0\\0\\0\\1\end{bmatrix} = \begin{bmatrix}0\\0\\\alpha\\\beta\end{bmatrix}$$

Any vector of the form $[0, 0, \alpha, \beta]$ with $\alpha + \beta = 1$ satisfies $A\mathbf{v} = \mathbf{v}$ and is a valid steady state. The specific steady state the system converges to depends on the initial condition $\mathbf{x}_0$.

In [ ]:
# Verify x_inf is a lambda=1 eigenvector
print('x_inf =')
print(np.round(x_inf.reshape(-1, 1), 4))
print()
print('A @ x_inf =')
print(np.round((A @ x_inf).reshape(-1, 1), 4))
print()
print('Equal:', np.allclose(A @ x_inf, x_inf, atol=1e-6))
print()

# Verify x_inf is a linear combination of the two lambda=1 eigenvectors
alpha = x_inf[2]   # R component
beta  = x_inf[3]   # D component
reconstructed = alpha * np.array([0,0,1,0]) + beta * np.array([0,0,0,1])
print(f'x_inf = {alpha:.4f} * [0,0,1,0]^T + {beta:.4f} * [0,0,0,1]^T')
print()
print('Reconstructed =')
print(np.round(reconstructed.reshape(-1, 1), 4))

### Different Initial Conditions, Different Steady States

The initial condition $\mathbf{x}_0$ determines which point in the $\lambda=1$ eigenspace the system converges to. Starting with more people already recovered shifts the long-run balance toward $R$; a different mix shifts it toward $D$.

In [ ]:
initial_conditions = {
    'All susceptible      [1.0, 0.0, 0.0, 0.0]': np.array([1.0, 0.0, 0.0, 0.0]),
    'Half infected        [0.5, 0.5, 0.0, 0.0]': np.array([0.5, 0.5, 0.0, 0.0]),
    'Mostly recovered       [0.1, 0.1, 0.8, 0.0]': np.array([0.1, 0.1, 0.8, 0.0]),
}

print(f'{"Initial condition":<45} {"R_inf":>8} {"D_inf":>8}  Is eigenvector (lam=1)?')
print('-' * 75)
for label, x0_i in initial_conditions.items():
    xi = np.linalg.matrix_power(A, 10000) @ x0_i
    is_eig = np.allclose(A @ xi, xi, atol=1e-6)
    print(f'{label:<45} {xi[2]:>8.4f} {xi[3]:>8.4f}  {is_eig}')

In [ ]:
# Run this cell without modification — show convergence from three starting points
fig, ax = plt.subplots(figsize=(9, 5))

ic_list = [
    (np.array([1.0, 0.0, 0.0, 0.0]), 'steelblue', 'x0 = [1.0, 0.0, 0.0, 0.0]'),
    (np.array([0.5, 0.5, 0.0, 0.0]), 'tomato',    'x0 = [0.5, 0.5, 0.0, 0.0]'),
    (np.array([0.1, 0.1, 0.8, 0.0]), 'seagreen', 'x0 = [0.1, 0.1, 0.8, 0.0]'),
]

T_plot = 200
for x0_i, color, label in ic_list:
    xi_inf = np.linalg.matrix_power(A, 10000) @ x0_i
    hist_i = np.zeros((4, T_plot+1))
    hist_i[:, 0] = x0_i
    xc = x0_i.copy()
    for t in range(T_plot):
        xc = A @ xc
        hist_i[:, t+1] = xc
    ax.plot(hist_i[2], color=color, lw=2,
            label=f'{label}  ->  R_inf={xi_inf[2]:.4f}, D_inf={xi_inf[3]:.4f}')
    ax.axhline(xi_inf[2], color=color, lw=1, linestyle='--', alpha=0.5)

ax.set_xlabel('Time step t')
ax.set_ylabel('Recovered fraction R_t')
ax.set_title('Different initial conditions converge to different steady states\n'
             '(all within the lam=1 eigenspace)')
ax.legend(fontsize=9)
ax.set_xlim(0, T_plot); ax.set_ylim(0, 1)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Question.** All three initial conditions produce a steady state with $S_\infty = 0$ and $I_\infty = 0$. Why must this always be true for this model, regardless of $\mathbf{x}_0$?

**Question.** The third initial condition already has 20% recovered at $t=0$. How does this shift the final $R_\infty$ compared to starting fully susceptible? Does the shift make intuitive sense?

**Question.** The non-unit eigenvalues (0.9671 and 0.8329) govern how fast the system decays toward the $\lambda=1$ eigenspace. The smaller the eigenvalue, the faster that component decays. Which eigenvalue controls the slowest-decaying transient — and what does that mean for how long the epidemic takes to resolve?

## 8.3 Modifying the Model

We now make two changes to the transition matrix and observe how each one alters the behavior of the epidemic. For each modification, we will build the new matrix, simulate the system, plot it against the original, and interpret the results.

### Modification 1: A More Contagious Variant

Suppose a new variant of the disease emerges that is three times as contagious. We model this by tripling the infection rate: instead of 5% of susceptible individuals becoming infected each step, 15% do. The remaining 85% stay susceptible.

All other rates stay the same — only the $S \to I$ and $S \to S$ entries change.

**Question.** Before running any code: which entries of $A$ change, and what do they become? Does the column-sum property still hold?

In [ ]:
A_variant = np.array([[0.85, 0.04, 0, 0],
                      [0.15, 0.85, 0, 0],
                      [0,    0.10, 1, 0],
                      [0,    0.01, 0, 1]])

print('Variant transition matrix:')
print(np.round(A_variant, 4))
print()
print('Column sums:', np.round(A_variant.sum(axis=0), 6))

In [ ]:
T_v = 200
history_v = np.zeros((4, T_v+1))
history_v[:, 0] = x0
x = x0.copy()
for t in range(T_v):
    x = A_variant @ x
    history_v[:, t+1] = x

x_inf_v = np.linalg.matrix_power(A_variant, 10000) @ x0
print('Variant steady state:')
for i, name in enumerate(labels_sird):
    print(f'  {name:<15}: {x_inf_v[i]:.4f}')

vals_v = np.linalg.eigvals(A_variant)
print()
print('Variant eigenvalues:', np.round(np.sort(np.real(vals_v))[::-1], 4))

In [ ]:
# Run this cell without modification
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
t_range = np.arange(T+1)

for ax, (hist, title) in zip(axes, [
    (history,   'Original (5% infection rate)'),
    (history_v, 'Variant  (15% infection rate)'),
]):
    for i in range(4):
        ax.plot(t_range, hist[i], color=colors_sird[i], lw=2, label=labels_sird[i])
    ax.set_xlabel('Time step $t$')
    ax.set_ylabel('Fraction of population')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(0, T); ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('SIRD Model: Original vs. More Contagious Variant', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** The peak infected fraction is much higher in the variant model. What is the approximate peak in each case, and at what time step does it occur?

**Question.** The long-run steady state is the same in both models — the same fraction eventually recovers and dies. Why? What property of the absorbing states guarantees this regardless of the infection rate?

**Question.** Look at the variant eigenvalues. The dominant non-unit eigenvalue is approximately 0.9275, compared to 0.9671 in the original. What does this tell you about the convergence speed of the variant model? Does the plot confirm this?

### Modification 2: Background Mortality

In the original model, susceptible individuals can only die from the disease — they must first become infected. Now suppose there is a small background mortality rate: 0.5% of susceptible individuals die per step from unrelated causes. This adds a direct flow from $S$ to $D$, reducing the $S \to S$ self-loop from 95% to 94.5%.

**Question.** Before running any code: which entries of $A$ change? Does this break the column-sum property?

In [ ]:
A_mortality = np.array([[0.945, 0.04, 0, 0],
                        [0.05,  0.85, 0, 0],
                        [0,     0.10, 1, 0],
                        [0.005, 0.01, 0, 1]])

print('Background mortality transition matrix:')
print(np.round(A_mortality, 4))
print()
print('Column sums:', np.round(A_mortality.sum(axis=0), 6))

In [ ]:
T_m = 300
history_m = np.zeros((4, T_m+1))
history_m[:, 0] = x0
x = x0.copy()
for t in range(T_m):
    x = A_mortality @ x
    history_m[:, t+1] = x

x_inf_m = np.linalg.matrix_power(A_mortality, 10000) @ x0
print('Background mortality steady state:')
for i, name in enumerate(labels_sird):
    print(f'  {name:<15}: {x_inf_m[i]:.4f}')
print()
print('Original deceased fraction:   ', round(x_inf[3], 4))
print('With background mortality:    ', round(x_inf_m[3], 4))

In [ ]:
# Run this cell without modification
t_range_m = np.arange(T_m+1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (hist, t_max, title) in zip(axes, [
    (history,   T,   'Original (no background mortality)'),
    (history_m, T_m, 'Background mortality (0.5% S→D per step)'),
]):
    for i in range(4):
        ax.plot(np.arange(t_max+1), hist[i], color=colors_sird[i], lw=2, label=labels_sird[i])
    ax.set_xlabel('Time step $t$')
    ax.set_ylabel('Fraction of population')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(0, t_max); ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('SIRD Model: Original vs. Background Mortality', fontsize=13)
plt.tight_layout()
plt.show()

**Question.** The deceased fraction in the long run is much larger with background mortality. Why? Which additional flow in $A$ is responsible?

**Question.** The shape of the infected curve looks similar in both models, but the susceptible curve declines faster with background mortality. Explain why: what two mechanisms are now draining the susceptible compartment simultaneously?

**Question.** In both modifications, the long-run values of $S$ and $I$ are still 0. Does this always have to be true for any SIRD-like model? Can you construct a transition matrix where $S$ or $I$ has a nonzero long-run value — and if so, what structural change would you need to make?